# Coupled Model Test

Speedy + Slab Ocean Model

In [ ]:
import os
#os.environ["JAX_PLATFORM_NAME"] = "cpu"

import jax
print("JAX device:", jax.devices())
jax.config.update('jax_disable_jit', False) # Turn off JIT because of an issue in shortwave_radiation.py:169
jax.config.update("jax_debug_infs", True) # doesn't add any time since the saved time is otherwise spent getting the nodal quantities
jax.config.update("jax_debug_nans", False) # some physics fields might be nan

In [ ]:
import sys
from pathlib import Path

paths_check = [
    (Path(os.path.abspath(".")) / ".." / ".." / "jax-gcm").resolve(),
    (Path(os.path.abspath(".")) / "..").resolve(),
]

for module_path in paths_check:
    module_path = str(module_path)
    if module_path in sys.path:
        print("Path exist: ", module_path)
    else:
        print("Add Path: ", module_path)
        sys.path.append(module_path)


In [ ]:
import numpy as np
import xarray as xr
import pandas as pd

import jcm
import jax_esm
from jax_esm.coupling.coupler import Coupler
from jax_esm.components.base import ComponentConfig
from jax_esm.components.JCM import JCM
from jax_esm.components.SlabOceanModel import SlabOceanModel
from jax_esm.components.FluxModel import FluxModel

## Create Boundary File

In [ ]:
from pathlib import Path
from jcm.model import get_coords
from jcm.boundaries import boundaries_from_file

# Prepare boundary file
boundary_file = (Path(jcm.__file__).parent / "data/bc/t30/clim/boundaries_daily.nc").resolve()

if not boundary_file.exists():
    print("Boundary file %s does not exist. Need to produce it." % (str(boundary_file),))
    import subprocess, sys
    interpolation_file = boundary_file.parent / "interpolate.py"
    subprocess.run([sys.executable, str(interpolation_file)], check=True)

if boundary_file.exists():
    print("Boundary file %s exists!" % (str(boundary_file), ) )
else:
    raise Exception("Something went wrong. The daily file is not generated. Please check.")

boundaries = boundaries_from_file(
    boundary_file,
    get_coords().horizontal
)

## Create Model

Currently, the time steps are not checked to be consistent. Should be checked in the future.

In [ ]:
start_dt = pd.Timestamp("2001-01-01")

sim_start_time = 0.0
total_simulation_time  =  20 * 86400.0 # sec
sim_end_time = sim_start_time + total_simulation_time

coupler_time_step      =  86400.0  # sec
coupler_steps          =  int(total_simulation_time / coupler_time_step)

atm_substeps          =  12           # count
atm_save_interval     =  coupler_time_step / atm_substeps # coupler_time_step # sec

ocn_substeps          =  1            # count
ocn_save_interval     = 24 *  3600.0  # sec

flx_substeps          =  1            # count
flx_save_interval     = 24 *  3600.0  # sec

In [ ]:
# Coupler Config
config_master = dict(
    total_simulation_time = total_simulation_time,
    time_step = coupler_time_step,
)

# Atmosphere model
config_atm = ComponentConfig(
    name = "atm",
    start_dt = start_dt,
    timestep = coupler_time_step,
    substeps = atm_substeps,
    save_interval = atm_save_interval,
    grid = None,
    params = dict(
        boundaries = boundaries,
    ),
)

model_atm = JCM(
    config = config_atm,
)

# Ocean model
config_ocn = ComponentConfig(
    name = "ocn",
    start_dt = start_dt,
    timestep = coupler_time_step,
    substeps = ocn_substeps,
    save_interval = ocn_save_interval,
    grid = None,
    params = dict(
        coords = model_atm.model.coords,
        geometry = model_atm.model.geometry,
        relaxation_time = 86400.0 * 60,
        boundaries = boundaries,
        boundary_file = boundary_file,
    ),
)
model_ocn = SlabOceanModel(config_ocn)

# Flux model
config_flx = ComponentConfig(
    name = "flx",
    start_dt = start_dt,
    timestep = coupler_time_step,
    substeps = flx_substeps,
    save_interval = flx_save_interval,
    grid = None,
    params = dict(
        coords = model_atm.model.coords,
        geometry = model_atm.model.geometry,
        boundaries = boundaries,
        boundary_file = boundary_file,
    ),
)
model_flx = FluxModel(config_flx)

In [ ]:
coupler = Coupler(
    config = config_master,
    components = dict(
        flx = model_flx,
        atm = model_atm,
        ocn = model_ocn,
    ), 
)

init_cplstate = coupler.initialize()

# Plot initial conditions

In [ ]:
import xarray as xr

In [ ]:
prog_varnames = list(init_cplstate.atm.prog.__dataclass_fields__.keys())

# Examine manually what variables we can plot
for varname in prog_varnames:
    v = getattr(init_cplstate.atm.prog, varname)
    print(f"Keys in atm.prog = {varname:s} => ", v.shape )
    

In [ ]:
init_nsp = xr.DataArray(
    data = init_cplstate.atm.prog.normalized_surface_pressure,
    dims = ["lon", "lat"],
)
init_nsp.plot.contourf(x='lon', y='lat', cmap="gnuplot2", add_colorbar=True, levels=np.linspace(0.9, 1.1, 11), extends="both")

In [ ]:
init_temperature = xr.DataArray(
    data = init_cplstate.atm.prog.temperature,
    dims = ["z", "lon", "lat"],
)
(init_temperature-273.15).isel(z=0).plot.contourf(x='lon', y='lat', cmap="gnuplot2", add_colorbar=True, levels=np.arange(-2, 36, 2), extends="both")

In [ ]:
init_u = xr.DataArray(
    data = init_cplstate.atm.prog.u_wind,
    dims = ["z", "lon", "lat"],
)
init_u.isel(z=0).plot.contourf(x='lon', y='lat', cmap="gnuplot2", add_colorbar=True, extends="both")


In [ ]:
init_SST = xr.DataArray(
    data = init_cplstate.ocn.prog.T,
    dims = ["lon", "lat"],
)

(init_SST-273.15).plot.contourf(x='lon', y='lat', cmap="gnuplot2", add_colorbar=True, levels=np.arange(-2, 36, 2), extends="both")


# Run the Model

In [ ]:
import numpy as np

final_cpl_state, pred = coupler.run(
    init_cplstate = init_cplstate,
    start_time    = sim_start_time,
    end_time      = sim_end_time,
    timestep      = coupler_time_step,
    save_interval_steps = 1,
    jax_scan = True, 
)

In [ ]:
print("Convert data to xarray")
ds_dict = coupler.predictions_to_xarray(pred)

# Diagnostics Plots

In [ ]:
import matplotlib.pyplot as plt

## Ocean Model Diagnostics

In [ ]:
ocn_ds = ds_dict["ocn"]

In [ ]:
ocn_ds["mld"].plot(x='lon', y='lat', col="time", col_wrap=3)

In [ ]:

ax = (ocn_ds["T"]-273.15).plot.contourf(x='lon', y='lat', col="time", col_wrap=3, cmap="gnuplot2", levels=np.linspace(-2, 35, 21))
plt.suptitle("SST")


In [ ]:

ax = (ocn_ds["T"]-ocn_ds["T"].isel(time=0)).plot.contourf(x='lon', y='lat', col="time", col_wrap=3, cmap="bwr", levels=np.linspace(-1, 1, 21)*5)
plt.suptitle("SST")


In [ ]:
ax = (ocn_ds["time"]/86400).plot(x='time', marker="o", markersize=10)
plt.grid(True)

## Flux Diagnostics

In [ ]:
flx_ds = ds_dict["flx"]

In [ ]:
flx_ds["heatflx"].plot.contourf(x='lon', y='lat', col="time", levels=10, col_wrap=3)

## Verify What Atmosphere Model Sees

In [ ]:
import numpy as np
import jax
from jcm.model import Predictions
import xarray as xr

time_slice = slice(0, None, atm_substeps)

# This function seems to have trouble dealing with stacked data.
pred_ds = ds_dict["atm"]


In [ ]:
pred_ds["u_wind"].isel(level=2, time=time_slice).plot.contourf(x='lon', y='lat', col="time", col_wrap=3, cmap="bwr", levels=np.linspace(-1, 1, 21)*30)

In [ ]:
pred_ds["specific_humidity"].isel(level=0, time=time_slice).plot(x='lon', y='lat', col="time", col_wrap=3)

In [ ]:
pred_ds["surface_flux.tskin"].isel(time=time_slice).plot(x='lon', y='lat', col="time", col_wrap=3)

## Verify the atmosphere sees the same SST as ocean gives

In [ ]:
print("The result should be zero, or very small numbers.")

# Atmosphere see SST a coupled-step late because of the framework
daily_sst_atm = pred_ds["surface_flux.tskin"].isel(time=time_slice).isel(time=slice(1, None, None))
daily_sst_ocn = ocn_ds["T"].isel(time=slice(0, -1, None))

# The time coordniates do not have the same unit. So I drop them for now.
daily_sst_atm = daily_sst_atm.drop_vars("time")
daily_sst_ocn = daily_sst_ocn.drop_vars("time")

SST_diff = daily_sst_atm - daily_sst_ocn 
SST_diff.plot.contourf(x='lon', y='lat', col="time", col_wrap=3, cmap="bwr")